## NLTK: токенізація, POS-теги та лематизація

Приклад показує, як отримати токени, частини мови та леми для англійського речення.


In [1]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith("J"):
        return wordnet.ADJ
    elif treebank_tag.startswith("V"):
        return wordnet.VERB
    elif treebank_tag.startswith("N"):
        return wordnet.NOUN
    elif treebank_tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN


nltk.download("punkt")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("averaged_perceptron_tagger_eng")

text = "The students are studying."

tokens = word_tokenize(text)
tagged_tokens = pos_tag(tokens)

lemmatizer = WordNetLemmatizer()

for word, tag in tagged_tokens:
    pos = get_wordnet_pos(tag)
    lemma = lemmatizer.lemmatize(word, pos)
    print(f"{word} (tag: {tag}) → {lemma}")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Student\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Student\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Student\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Student\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.


The (tag: DT) → The
students (tag: NNS) → student
are (tag: VBP) → be
studying (tag: VBG) → study
. (tag: .) → .


## spaCy: лематизація українського тексту

Приклад використовує українську модель spaCy та відфільтровує стоп-слова й пунктуацію.


In [3]:
import spacy

# en_core_web_sm - English
# uk_core_news_sm - Ukrainian
# fr_core_news_md - French

# web — trained on mixed web text (mostly English).
# news — trained primarily on news text (most other languages).

# sm — Small
# md — Medium (includes word vectors)
# lg — Large (larger word vectors)
# trf — Transformer-based (typically highest accuracy but slower)

# python -m spacy download uk_core_news_sm
nlp = spacy.load("en_core_web_sm") #, "uk_core_news_sm")


text = "Студенти швидко навчаються та написали кілька програм."
text = "Students are studying very fast and have already written several apps."

doc = nlp(text)

for token in doc:
    if not token.is_stop and not token.is_punct:
        print(f"{token.text:15} → {token.lemma_} ({token.pos_})")


OSError: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

## tiktoken: токени для моделей OpenAI

Приклад демонструє кодування, декодування та перегляд окремих токенів.


In [ ]:
import tiktoken


# r50k_base - gpt-3
# p50k_base - codex
# cl100k_base - GPT-3.5, GPT-4
# o200k_base - сучасні GPT-4o та GPT-5

encoding = tiktoken.encoding_for_model("gpt-5")

tokens = encoding.encode("Hello world!")

print(tokens)

text = encoding.decode(tokens)

print(text)

text = """
This is a very long prompt...
"""

num_tokens = len(encoding.encode(text))

print(num_tokens)


text = "internationalization"

tokens = encoding.encode(text)

print(tokens)

for token in tokens:
    piece = encoding.decode([token])
    print(f"{token:>6} -> {repr(piece)}")



text = "машинне навчання"

tokens = encoding.encode(text)

for token in tokens:
    print(token, repr(encoding.decode([token])))


## Sentence Transformers: embeddings і cosine similarity

Приклад створює embeddings для речень і порівнює їх через cosine similarity.


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# paraphrase: модель навчали так, щоб речення з однаковим змістом мали близькі вектори;
# multilingual: працює більш ніж зі 50 мовами;
# MiniLM: це невелика трансформерна модель;
# L12: має 12 Transformer Encoder блоків.

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

embedding = model.encode("Кіт спить на дивані")

print(len(embedding))
print(embedding[:10])

texts = [
    "Кіт спить на дивані",
    "На дивані спить кішка",
    "Сьогодні йде сильний дощ"
]

embeddings = model.encode(texts)

print(cosine_similarity(embeddings))

# |       |      Кіт |    Кішка |      Дощ |
# | ----- | -------: | -------: | -------: |
# | Кіт   | **1.00** | **0.93** |     0.18 |
# | Кішка | **0.93** | **1.00** |     0.21 |
# | Дощ   |     0.18 |     0.21 | **1.00** |


## Semantic search: пошук найближчого документа

Джерело: `semantic_search_sample.py`

Приклад нормалізує embeddings і знаходить документ з найбільшою схожістю до запиту.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np


documents = [
    "The product quality is excellent and the design is modern.",
    "The delivery was late and the package was damaged.",
    "Customer support helped me reset my account password.",
    "The price is high but the product works well.",
    "The mobile app crashes during payment.",
]

queries = [
    "I need help with my password.",
    "The item arrived too late.",
    "The app has a payment bug.",
]

model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = model.encode(documents, normalize_embeddings=True)

for query in queries:
    query_embedding = model.encode([query], normalize_embeddings=True)
    print("query_embedding.shape:", query_embedding.shape)
    print("doc_embeddings.shape:", doc_embeddings.shape)
    print("doc_embeddings.T.shape:", doc_embeddings.T.shape)
    scores = query_embedding @ doc_embeddings.T
    top_idx = int(np.argmax(scores[0]))

    print("Query:", query)
    print("Best match:", documents[top_idx])
    print("Score:", float(scores[0][top_idx]))
    print("-" * 80)
